# Transformer 架构全面解析

> 对应论文：*Attention Is All You Need*（Vaswani et al., 2017）——改变 NLP 历史的论文

本教程是目前中文最详细的 Transformer 教程之一。你将从零理解每个组件的**设计动机**、**数学推导**、**NumPy 手动实现**，最后搭建出一个可以生成文本的微型 GPT。

## 学习目标
- 理解为什么 RNN 不够好，Transformer 解决了什么问题
- 从零实现：位置编码、Self-Attention、MHA、FFN、LayerNorm、完整 Decoder Block
- 实现自回归文本生成（Greedy / Temperature / Top-k / Top-p）
- 理解 GPT 的训练目标：Next Token Prediction + Cross-Entropy Loss
- 用 PyTorch 验证手动实现的正确性

## 依赖
```python
pip install numpy matplotlib torch
```
无需 API Key，纯本地运行。

## 第一章：为什么需要 Transformer？

语言模型的核心任务：**给定前文，预测下一个词**。

要做好这件事，模型必须能捕捉句子中任意两个词的依赖关系——例如「The animal didn't cross the street because **it** was too tired」中，"it" 指的是 "animal" 而非 "street"，跨度超过 5 个词。

**RNN 的两大死穴：**
- **串行依赖**：h(t) = f(h(t-1), x(t))，必须等前一步结束才能算下一步，GPU 几千个核心全部空转
- **梯度消失**：信息经过 N 个节点传播，梯度按乘法规则衰减，长距离依赖在训练中几乎学不到

**Transformer 的答案：** 自注意力让每个 token **直接**与所有其他 token 交互，依赖路径长度恒为 O(1)，整个序列并行计算。代价是 O(n²·d) 的显存开销——这也是为什么长上下文推理很贵。

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 760 220" width="760" height="220" style="font-family:sans-serif;background:#fafafa;border-radius:8px;">
  <!-- Title -->
  <text x="380" y="22" text-anchor="middle" font-size="13" font-weight="bold" fill="#333">RNN vs Transformer：串行 vs 并行</text>

  <!-- === RNN 区域 === -->
  <rect x="20" y="35" width="330" height="170" rx="8" fill="#fff3e0" stroke="#ffb74d" stroke-width="1.5"/>
  <text x="185" y="55" text-anchor="middle" font-size="11.5" font-weight="bold" fill="#e65100">RNN — 串行，O(n) 步</text>

  <!-- RNN nodes -->
  <!-- h1 -->
  <circle cx="60"  cy="110" r="22" fill="#ffcc80" stroke="#ff9800" stroke-width="1.5"/>
  <text x="60"  cy="110" text-anchor="middle" dominant-baseline="middle" font-size="10" fill="#333">
    <tspan x="60" dy="-5">h₁</tspan><tspan x="60" dy="12">The</tspan>
  </text>
  <!-- h2 -->
  <circle cx="120" cy="110" r="22" fill="#ffcc80" stroke="#ff9800" stroke-width="1.5"/>
  <text x="120" text-anchor="middle" dominant-baseline="middle" font-size="10" fill="#333">
    <tspan x="120" y="105">h₂</tspan><tspan x="120" y="117">animal</tspan>
  </text>
  <!-- h3 -->
  <circle cx="180" cy="110" r="22" fill="#ffcc80" stroke="#ff9800" stroke-width="1.5"/>
  <text x="180" text-anchor="middle" font-size="10" fill="#333">
    <tspan x="180" y="105">h₃</tspan><tspan x="180" y="117">didn't</tspan>
  </text>
  <!-- h4 -->
  <circle cx="240" cy="110" r="22" fill="#ffcc80" stroke="#ff9800" stroke-width="1.5"/>
  <text x="240" text-anchor="middle" font-size="10" fill="#333">
    <tspan x="240" y="105">h₄</tspan><tspan x="240" y="117">cross</tspan>
  </text>
  <!-- h5 -->
  <circle cx="300" cy="110" r="22" fill="#ffcc80" stroke="#ff9800" stroke-width="1.5"/>
  <text x="300" text-anchor="middle" font-size="10" fill="#333">
    <tspan x="300" y="105">h₅</tspan><tspan x="300" y="117">street</tspan>
  </text>
  <!-- Arrows h1->h2->h3->h4->h5 -->
  <defs>
    <marker id="arr" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto">
      <path d="M0,0 L0,6 L8,3 z" fill="#ff9800"/>
    </marker>
    <marker id="arr2" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto">
      <path d="M0,0 L0,6 L8,3 z" fill="#42a5f5"/>
    </marker>
    <marker id="arr3" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto">
      <path d="M0,0 L0,6 L8,3 z" fill="#999"/>
    </marker>
  </defs>
  <line x1="83" y1="110" x2="96" y2="110" stroke="#ff9800" stroke-width="1.5" marker-end="url(#arr)"/>
  <line x1="143" y1="110" x2="156" y2="110" stroke="#ff9800" stroke-width="1.5" marker-end="url(#arr)"/>
  <line x1="203" y1="110" x2="216" y2="110" stroke="#ff9800" stroke-width="1.5" marker-end="url(#arr)"/>
  <line x1="263" y1="110" x2="276" y2="110" stroke="#ff9800" stroke-width="1.5" marker-end="url(#arr)"/>

  <!-- RNN labels -->
  <text x="185" y="155" text-anchor="middle" font-size="10" fill="#bf360c">❌ 无法并行　❌ 梯度消失</text>
  <text x="185" y="170" text-anchor="middle" font-size="9.5" fill="#bf360c">h₁ 要传过 4 步才能影响 h₅</text>

  <!-- === Transformer 区域 === -->
  <rect x="410" y="35" width="330" height="170" rx="8" fill="#e3f2fd" stroke="#42a5f5" stroke-width="1.5"/>
  <text x="575" y="55" text-anchor="middle" font-size="11.5" font-weight="bold" fill="#1565c0">Transformer — 并行，O(1) 步</text>

  <!-- TF nodes (1 row) -->
  <rect x="425" y="85" width="38" height="30" rx="4" fill="#90caf9" stroke="#1976d2" stroke-width="1"/>
  <text x="444" y="105" text-anchor="middle" font-size="9.5" fill="#0d47a1">The</text>

  <rect x="475" y="85" width="38" height="30" rx="4" fill="#90caf9" stroke="#1976d2" stroke-width="1"/>
  <text x="494" y="105" text-anchor="middle" font-size="9" fill="#0d47a1">animal</text>

  <rect x="525" y="85" width="38" height="30" rx="4" fill="#90caf9" stroke="#1976d2" stroke-width="1"/>
  <text x="544" y="105" text-anchor="middle" font-size="9" fill="#0d47a1">didn't</text>

  <rect x="575" y="85" width="38" height="30" rx="4" fill="#90caf9" stroke="#1976d2" stroke-width="1"/>
  <text x="594" y="105" text-anchor="middle" font-size="9.5" fill="#0d47a1">cross</text>

  <rect x="625" y="85" width="38" height="30" rx="4" fill="#90caf9" stroke="#1976d2" stroke-width="1"/>
  <text x="644" y="105" text-anchor="middle" font-size="9.5" fill="#0d47a1">the</text>

  <rect x="675" y="85" width="38" height="30" rx="4" fill="#64b5f6" stroke="#1565c0" stroke-width="1.5"/>
  <text x="694" y="105" text-anchor="middle" font-size="9.5" font-weight="bold" fill="#0d47a1">street</text>

  <!-- Attention arc: animal <-> street (bidirectional, curved) -->
  <path d="M494,85 Q544,42 694,85" fill="none" stroke="#1565c0" stroke-width="1.5" stroke-dasharray="5,3" marker-end="url(#arr2)"/>
  <text x="594" y="50" text-anchor="middle" font-size="9" fill="#1565c0">直接注意力（1步）</text>

  <!-- All-to-all indicator -->
  <text x="575" y="148" text-anchor="middle" font-size="10" fill="#1565c0">✓ 全部词对并行计算</text>
  <text x="575" y="163" text-anchor="middle" font-size="9.5" fill="#0d47a1">任意两词直接交互，无距离衰减</text>

  <!-- Arrow between panels -->
  <line x1="355" y1="120" x2="405" y2="120" stroke="#999" stroke-width="1.5" stroke-dasharray="4,3" marker-end="url(#arr3)"/>
  <text x="380" y="113" text-anchor="middle" font-size="9" fill="#777">改进</text>
</svg>

## 第二章：完整的文本处理流水线

在深入各个组件之前，先看清楚文本是怎么一步步变成模型可以处理的数字的：

```
原始文本: "I love cats"
     ↓ 分词（Tokenization）
Token IDs: [40, 1842, 11875]
     ↓ 嵌入查表（Embedding Lookup）
嵌入矩阵: shape (3, d_model)  ← 每个 token → 一个 d_model 维向量
     ↓ 位置编码（Positional Encoding）
位置感知嵌入: shape (3, d_model)  ← 叠加位置信息
     ↓ N 个 Decoder Block
上下文化表示: shape (3, d_model)  ← 每个 token 知道了整个上下文
     ↓ 输出头（Linear + Softmax）
概率分布: shape (3, vocab_size)   ← 每个位置预测下一个 token
```

**关键直觉：** 整个流程中，tensor 的 shape 是 (batch, seq_len, d_model)。
这个 shape **始终不变**，直到最后的输出头。
每个 Decoder Block 是「输入什么 shape，输出什么 shape」的变换。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')
matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

# ============================================================
# 一个玩具例子贯穿全文：词表大小 16，序列长度 6，d_model=8
# ============================================================
VOCAB_SIZE = 16
SEQ_LEN    = 6
D_MODEL    = 8

np.random.seed(42)

# 词表（演示用）
vocab = {
    '<PAD>': 0, '<BOS>': 1, '<EOS>': 2,
    'the': 3, 'cat': 4, 'sat': 5,
    'on': 6, 'mat': 7, 'dog': 8,
    'ran': 9, 'fast': 10, 'slow': 11,
    'a': 12, 'big': 13, 'small': 14, 'is': 15
}
id2word = {v: k for k, v in vocab.items()}

# 模拟一个句子: "<BOS> the cat sat on mat"
token_ids = [vocab['<BOS>'], vocab['the'], vocab['cat'],
             vocab['sat'], vocab['on'], vocab['mat']]
print("原始 token IDs:", token_ids)
print("对应词：", [id2word[t] for t in token_ids])

# ---- Embedding 查找表 ----
# 实际中由模型学习，这里用随机初始化
embedding_table = np.random.randn(VOCAB_SIZE, D_MODEL) * 0.1
print(f"\nEmbedding Table shape: {embedding_table.shape}")
print(f"  → 词表中每个 token 对应一个 {D_MODEL} 维向量")

# 根据 token_ids 查表
X = embedding_table[token_ids]  # shape: (SEQ_LEN, D_MODEL)
print(f"\n序列嵌入 X shape: {X.shape}  (6 个 token，每个 8 维)")
print(f"第一个 token '<BOS>' 的向量: {X[0].round(3)}")

## 第三章：位置编码（Positional Encoding）

### 3.1 问题：自注意力是「置换不变」的

如果我们打乱输入序列的顺序，自注意力的输出也会以相同方式打乱——它**不感知顺序**。

但语言依赖顺序：
- 「猫追狗」≠「狗追猫」
- 「我没有不高兴」和「我不高兴没有」意思完全不同

### 3.2 正弦位置编码（论文原版）

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

**设计动机：**
- `pos` 从 0 到 max_seq_len-1，每个位置有唯一的编码「指纹」
- `i` 控制频率：低维度 i 小 → 高频正弦（区分近距离位置）；高维度 i 大 → 低频正弦（区分远距离位置）
- 10000 这个基数让编码在各个频率上分布均匀
- 数学性质：$PE_{pos+k}$ 可以表示为 $PE_{pos}$ 的线性变换，因此模型可以学习**相对位置**

### 3.3 为什么不用简单的 [0, 1, 2, ...] 整数？
- 归一化到 [0,1] 后，不同长度序列的步长不同，泛化性差
- 整数随序列增大而增大，没有上界，可能影响训练稳定性
- 正弦编码是有界的（[-1, 1]），且包含连续的梯度信息

### 3.4 现代替代方案
| 方案 | 使用模型 | 优势 |
|------|---------|------|
| 正弦固定编码 | 原始 Transformer | 无需学习，可外推 |
| 可学习位置编码 | GPT-2, BERT | 灵活，但不能外推 |
| RoPE（旋转位置编码） | LLaMA, GPT-NeoX | 相对位置，长度外推能力强 |
| ALiBi | BLOOM | 线性注意力偏置，外推最强 |

In [ ]:
def positional_encoding(max_seq_len, d_model):
    """
    正弦位置编码实现
    
    Args:
        max_seq_len: 最大序列长度
        d_model: 嵌入维度
    Returns:
        PE: shape (max_seq_len, d_model)
    """
    PE = np.zeros((max_seq_len, d_model))
    
    # pos: (max_seq_len, 1), i: (1, d_model/2)
    pos = np.arange(max_seq_len)[:, np.newaxis]
    i   = np.arange(0, d_model, 2)[np.newaxis, :]
    
    # 频率衰减项：随着维度 i 增大，频率降低（波长增大）
    div_term = np.power(10000.0, i / d_model)  # shape: (1, d_model/2)
    
    PE[:, 0::2] = np.sin(pos / div_term)  # 偶数维度用 sin
    PE[:, 1::2] = np.cos(pos / div_term)  # 奇数维度用 cos
    return PE

# 生成位置编码
PE = positional_encoding(max_seq_len=50, d_model=64)

# ---- 可视化 ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 图1：热力图（每行=一个位置的完整编码向量）
im = axes[0].imshow(PE, aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
axes[0].set_xlabel('Embedding 维度 (d_model=64)')
axes[0].set_ylabel('序列位置 pos')
axes[0].set_title('位置编码热力图\n（每行=一个位置的唯一「指纹」）')
plt.colorbar(im, ax=axes[0])

# 图2：不同维度的波形（展示频率差异）
for dim, color in zip([0, 2, 8, 32], ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']):
    axes[1].plot(PE[:, dim], color=color, label=f'dim={dim}')
axes[1].set_xlabel('序列位置 pos')
axes[1].set_ylabel('编码值')
axes[1].set_title('不同维度的波形\n（低维高频，高维低频）')
axes[1].legend()
axes[1].grid(alpha=0.3)

# 图3：位置向量间的余弦相似度（验证：相邻位置更相似）
positions = [0, 1, 2, 5, 10, 20]
sim_matrix = np.zeros((len(positions), len(positions)))
for i, p1 in enumerate(positions):
    for j, p2 in enumerate(positions):
        v1, v2 = PE[p1], PE[p2]
        sim_matrix[i, j] = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

im2 = axes[2].imshow(sim_matrix, cmap='coolwarm', vmin=-0.2, vmax=1)
axes[2].set_xticks(range(len(positions)))
axes[2].set_yticks(range(len(positions)))
axes[2].set_xticklabels(positions)
axes[2].set_yticklabels(positions)
axes[2].set_title('不同位置编码的余弦相似度\n（对角线=1，距离越远越不相似）')
plt.colorbar(im2, ax=axes[2])
for i in range(len(positions)):
    for j in range(len(positions)):
        axes[2].text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=8)

plt.suptitle('正弦位置编码分析', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('pe_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

print("关键观察：")
print(f"  pos=0 和 pos=1 的余弦相似度: {sim_matrix[0,1]:.3f}  （相邻，很相似）")
print(f"  pos=0 和 pos=10 的余弦相似度: {sim_matrix[0,4]:.3f} （较远，相似度下降）")
print(f"  pos=0 和 pos=20 的余弦相似度: {sim_matrix[0,5]:.3f} （很远，相似度更低）")

In [ ]:
# 将位置编码加到序列嵌入上
PE_seq = positional_encoding(SEQ_LEN, D_MODEL)
X_with_pos = X + PE_seq  # 直接相加，shape 不变

print(f"嵌入向量 X shape: {X.shape}")
print(f"位置编码 PE shape: {PE_seq.shape}")
print(f"相加结果 shape: {X_with_pos.shape}  ← 形状不变！")
print()
print("前两个 token 的嵌入（加位置编码前 vs 后）:")
print(f"  token[0] 原始: {X[0].round(3)}")
print(f"  token[0] PE:   {PE_seq[0].round(3)}")
print(f"  token[0] 相加: {X_with_pos[0].round(3)}")
print()
print("注：dropout 通常在加完位置编码后应用（训练时随机归零部分神经元，防过拟合）")

## 第四章：自注意力（Self-Attention）深度解析

### 4.1 核心直觉

问题：对于句子 "The animal didn't cross the street because **it** was too tired"，
模型如何知道 "it" 指的是 "animal" 而不是 "street"？

自注意力的答案：让每个词「问」所有其他词：「你和我相关吗？」

- **Q（Query）**：「我在寻找什么类型的信息？」
- **K（Key）**：「我能提供什么类型的信息？」  
- **V（Value）**：「如果你找到我了，你实际得到的内容是什么？」

类比搜索引擎：
- Query = 搜索关键词（"it 指代什么实体？"）
- Key = 每篇文章的标签（"animal: 生命体", "street: 地点"）
- Value = 文章的实际内容
- 搜索结果 = 与 Query 最匹配的几篇文章的内容加权

### 4.2 完整数学推导

**Step 1：线性投影**

$$Q = XW^Q, \quad K = XW^K, \quad V = XW^V$$

- X: (seq_len, d_model)，输入序列
- W^Q, W^K, W^V: (d_model, d_k)，可学习投影矩阵
- Q, K, V: (seq_len, d_k)

**Step 2：计算注意力分数**

$$\text{score}_{ij} = \frac{q_i \cdot k_j}{\sqrt{d_k}}$$

- $q_i$：第 i 个 token 的查询向量
- $k_j$：第 j 个 token 的键向量
- 点积衡量两个向量的「相关程度」

**Step 3：为什么除以 √d_k？**

假设 $q_i, k_j$ 的每个分量均值为0，方差为1：
- $q_i \cdot k_j = \sum_{l=1}^{d_k} q_{il} k_{jl}$
- 方差 = $d_k$（独立同分布随机变量相加）
- 标准差 = $\sqrt{d_k}$

不除以 $\sqrt{d_k}$，当 $d_k$ 很大（如 64）时，点积可能很大（如 ±8），
导致 softmax 输出几乎是 one-hot，梯度接近 0（梯度消失）。

除以 $\sqrt{d_k}$ 后，点积方差归一化回 1，softmax 梯度正常。

**Step 4：Softmax 归一化**

$$\alpha_{ij} = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)_{ij}$$

每行的权重之和为 1，可理解为「注意力概率分布」。

**Step 5：加权求和**

$$\text{output}_i = \sum_j \alpha_{ij} v_j$$

第 i 个 token 的输出 = 所有 token 的值向量的加权平均，权重由注意力分数决定。

In [ ]:
def softmax(x, axis=-1):
    """数值稳定的 softmax（减去最大值防止 exp 溢出）"""
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None, return_weights=True):
    """
    缩放点积注意力
    
    Args:
        Q: (seq_len_q, d_k)
        K: (seq_len_k, d_k)
        V: (seq_len_k, d_v)
        mask: (seq_len_q, seq_len_k)，True 表示「遮盖（设为 -inf）」
    Returns:
        output: (seq_len_q, d_v)
        weights: (seq_len_q, seq_len_k)
    """
    d_k = Q.shape[-1]
    
    # Step 1+2: 计算注意力分数（QK^T / √d_k）
    scores = Q @ K.T / np.sqrt(d_k)   # (seq_len_q, seq_len_k)
    
    # Step 3: 应用掩码
    if mask is not None:
        scores = np.where(mask, -1e9, scores)  # 遮盖位置设为极小值
    
    # Step 4: Softmax
    weights = softmax(scores, axis=-1)         # 行归一化
    
    # Step 5: 加权求和
    output = weights @ V                       # (seq_len_q, d_v)
    
    return (output, weights) if return_weights else output

# ====================== 关键实验：验证 √d_k 缩放的必要性 ======================
print("=== 实验：展示 √d_k 缩放的重要性 ===\n")
np.random.seed(0)
d_k = 64
q = np.random.randn(d_k)
k_list = np.random.randn(10, d_k)

# 未缩放
raw_scores = k_list @ q
print(f"未缩放的点积分数范围: [{raw_scores.min():.2f}, {raw_scores.max():.2f}]")
print(f"  std={raw_scores.std():.2f}（预期≈√d_k={np.sqrt(d_k):.2f}）")
print(f"  Softmax 结果（容易变成 one-hot）: {softmax(raw_scores).round(3)}")

# 缩放后
scaled_scores = raw_scores / np.sqrt(d_k)
print(f"\n缩放后的分数范围: [{scaled_scores.min():.2f}, {scaled_scores.max():.2f}]")
print(f"  std={scaled_scores.std():.2f}（≈1，正常范围）")
print(f"  Softmax 结果（更均匀的分布）: {softmax(scaled_scores).round(3)}")

In [ ]:
# ====================== 具体例子：句子中的注意力 ======================
# 手动构建一个简单的 Q, K, V 来展示注意力的语义含义
# 词：<BOS> the  cat  sat  on  mat
#  ID:  1    3    4    5    6    7

# 用真实的随机投影矩阵
np.random.seed(42)
d_k = D_MODEL
W_q = np.random.randn(D_MODEL, d_k) * 0.3
W_k = np.random.randn(D_MODEL, d_k) * 0.3
W_v = np.random.randn(D_MODEL, d_k) * 0.3

Q = X_with_pos @ W_q   # (6, 8)
K = X_with_pos @ W_k
V = X_with_pos @ W_v

output, attn_weights = scaled_dot_product_attention(Q, K, V)

print("注意力权重矩阵 (行=Query位置，列=Key位置):")
print("位置对应词：", [id2word[t] for t in token_ids])
print()
header = "     " + "  ".join(f"{id2word[t]:^6}" for t in token_ids)
print(header)
for i, tok_id in enumerate(token_ids):
    row = f"{id2word[tok_id]:^5}" + "  ".join(f"{attn_weights[i,j]:.3f}" for j in range(SEQ_LEN))
    print(row)

print(f"\n每行权重之和（应为1.0）: {attn_weights.sum(axis=1).round(4)}")

# 可视化
fig, ax = plt.subplots(figsize=(8, 6))
words = [id2word[t] for t in token_ids]
im = ax.imshow(attn_weights, cmap='Blues', vmin=0, vmax=attn_weights.max())
ax.set_xticks(range(SEQ_LEN))
ax.set_yticks(range(SEQ_LEN))
ax.set_xticklabels(words, fontsize=12)
ax.set_yticklabels(words, fontsize=12)
ax.set_xlabel('Key（被关注的位置）', fontsize=12)
ax.set_ylabel('Query（主动关注的位置）', fontsize=12)
ax.set_title('自注意力权重矩阵（无掩码）\n颜色越深=关注度越高', fontsize=13)
for i in range(SEQ_LEN):
    for j in range(SEQ_LEN):
        ax.text(j, i, f'{attn_weights[i,j]:.2f}', 
                ha='center', va='center', fontsize=9,
                color='white' if attn_weights[i,j] > 0.25 else 'black')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig('attention_weights.png', dpi=120, bbox_inches='tight')
plt.show()

## 第五章：因果掩码（Causal Mask）

### 5.1 为什么需要掩码？

在**训练**阶段，我们给模型看整个句子，然后让它预测每个位置的下一个词：
- 给「the」，预测「cat」
- 给「the cat」，预测「sat」
- 给「the cat sat」，预测「on」
- ...

这本可以一次并行完成，但有个问题：**预测位置 3 时，不能让模型看到位置 4, 5, 6 的答案！**（信息泄露）

因此需要**因果掩码**（Causal Mask / Autoregressive Mask）：
```
         Key 位置（被关注）
         0    1    2    3    4    5
Query 0: [√    ×    ×    ×    ×    ×]  ← <BOS> 只看自己
      1: [√    √    ×    ×    ×    ×]  ← "the" 看 <BOS> 和自己
      2: [√    √    √    ×    ×    ×]  ← "cat" 看前两个和自己
      3: [√    √    √    √    ×    ×]  ← "sat" 看前三个和自己
      4: [√    √    √    √    √    ×]
      5: [√    √    √    √    √    √]  ← "mat" 看所有位置
```

实现方式：把上三角位置的分数设为 -∞，Softmax 后这些位置的权重变为 0。

### 5.2 推理阶段呢？

推理时是**逐步生成**的：
1. 输入「<BOS>」，模型输出下一词的概率 → 采样得「the」
2. 输入「<BOS> the」，模型输出下一词概率 → 采样得「cat」
3. 以此类推...

这时天然就没有「偷看未来」的问题，掩码只是让训练和推理行为一致。

In [ ]:
def make_causal_mask(seq_len):
    """创建因果掩码：True 表示「遮盖（不可见）」"""
    # 上三角（不含对角线）= True（遮盖）
    mask = ~np.tril(np.ones((seq_len, seq_len), dtype=bool))
    return mask

causal_mask = make_causal_mask(SEQ_LEN)
print("因果掩码矩阵（True=遮盖，False=可见）:")
words = [id2word[t] for t in token_ids]
header = "          " + "  ".join(f"{w:^6}" for w in words)
print(header)
for i, word in enumerate(words):
    row = f"{word:^10}" + "  ".join("遮盖  " if causal_mask[i,j] else "可见  " for j in range(SEQ_LEN))
    print(row)

# 对比：有无掩码的注意力权重
output_causal, weights_causal = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, w, title in zip(axes, [attn_weights, weights_causal], 
                          ['无掩码（Encoder 风格）', '因果掩码（Decoder/GPT 风格）']):
    im = ax.imshow(w, cmap='Blues', vmin=0, vmax=max(attn_weights.max(), weights_causal.max()))
    ax.set_xticks(range(SEQ_LEN))
    ax.set_yticks(range(SEQ_LEN))
    ax.set_xticklabels(words, fontsize=10)
    ax.set_yticklabels(words, fontsize=10)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Key 位置')
    ax.set_ylabel('Query 位置')
    for i in range(SEQ_LEN):
        for j in range(SEQ_LEN):
            ax.text(j, i, f'{w[i,j]:.2f}', ha='center', va='center', fontsize=8,
                    color='white' if w[i,j] > 0.3 else 'black')
plt.suptitle('有无因果掩码的对比（注意右图上三角全为0.00）', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('causal_mask_compare.png', dpi=120, bbox_inches='tight')
plt.show()
print("\n关键：加掩码后，每个位置只关注自己和之前的位置，上三角权重精确为0")

## 第六章：多头注意力（Multi-Head Attention, MHA）

### 6.1 为什么需要多个头？

单头注意力在 d_k 维空间中只能捕获「一种」相关模式。
但语言中存在多种依赖关系：
- **语法依赖**：主语-谓语-宾语
- **语义关联**：同义词、反义词
- **共指关系**：代词指代
- **局部上下文**：相邻词的搭配
- ...

多头注意力：**将 d_model 维空间分成 h 个子空间，每个头独立学习不同类型的依赖**。

### 6.2 数学形式

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h) W^O$$
$$\text{head}_i = \text{Attention}(QW^Q_i, KW^K_i, VW^V_i)$$

- 每个头：$W^Q_i \in \mathbb{R}^{d_{model} \times d_k}$，$d_k = d_{model}/h$
- 所有头拼接后：$(seq\_len, h \cdot d_k) = (seq\_len, d_{model})$
- 再经过 $W^O \in \mathbb{R}^{d_{model} \times d_{model}}$ 输出投影

### 6.3 参数量分析

设 d_model=512, h=8, d_k=64:
- 每个头的 W_Q, W_K, W_V: 512×64 = 32,768 × 3
- 总 W_Q, W_K, W_V = 512×512 each（等价于合并在一起投影再分割）
- W_O: 512×512
- **总计：4 × 512² = 1,048,576 参数**
- 与单头（d_k=512）参数量相同，但表达能力更强！

### 6.4 实现技巧

实际实现中，不用真的分割成 h 个独立矩阵，而是：
1. 一次性计算 Q = X @ W_Q (seq_len, d_model)
2. reshape 成 (h, seq_len, d_k) 
3. 对 h 个头并行计算注意力
4. reshape 回来拼接

In [ ]:
def multi_head_attention(X, W_q, W_k, W_v, W_o, num_heads, mask=None):
    """
    多头注意力完整实现
    
    Args:
        X: (seq_len, d_model)
        W_q, W_k, W_v: (d_model, d_model)
        W_o: (d_model, d_model)
        num_heads: 头数
        mask: (seq_len, seq_len)，True=遮盖
    Returns:
        output: (seq_len, d_model)
        all_head_weights: list of (seq_len, seq_len)
    """
    seq_len, d_model = X.shape
    assert d_model % num_heads == 0, "d_model 必须能被 num_heads 整除"
    d_k = d_model // num_heads
    
    # ---- Step 1: 整体投影（比逐头投影效率高得多）----
    Q = X @ W_q   # (seq_len, d_model)
    K = X @ W_k
    V = X @ W_v
    
    # ---- Step 2: 分割成多个头 ----
    # (seq_len, d_model) -> (seq_len, num_heads, d_k) -> (num_heads, seq_len, d_k)
    Q = Q.reshape(seq_len, num_heads, d_k).transpose(1, 0, 2)
    K = K.reshape(seq_len, num_heads, d_k).transpose(1, 0, 2)
    V = V.reshape(seq_len, num_heads, d_k).transpose(1, 0, 2)
    # Q, K, V shape: (num_heads, seq_len, d_k)
    
    # ---- Step 3: 每个头独立计算注意力 ----
    head_outputs    = []
    all_head_weights = []
    for h_idx in range(num_heads):
        out_h, w_h = scaled_dot_product_attention(Q[h_idx], K[h_idx], V[h_idx], mask)
        head_outputs.append(out_h)      # (seq_len, d_k)
        all_head_weights.append(w_h)    # (seq_len, seq_len)
    
    # ---- Step 4: 拼接 + 输出投影 ----
    concat = np.concatenate(head_outputs, axis=-1)  # (seq_len, d_model)
    output = concat @ W_o                           # (seq_len, d_model)
    
    return output, all_head_weights

# 测试
NUM_HEADS = 4
np.random.seed(7)
scale = 0.1
W_q_mha = np.random.randn(D_MODEL, D_MODEL) * scale
W_k_mha = np.random.randn(D_MODEL, D_MODEL) * scale
W_v_mha = np.random.randn(D_MODEL, D_MODEL) * scale
W_o_mha = np.random.randn(D_MODEL, D_MODEL) * scale

mha_out, head_weights = multi_head_attention(
    X_with_pos, W_q_mha, W_k_mha, W_v_mha, W_o_mha, 
    num_heads=NUM_HEADS, mask=causal_mask
)
print(f"输入 shape: {X_with_pos.shape}")
print(f"MHA 输出 shape: {mha_out.shape}  （形状完全不变！）")
print(f"头数: {NUM_HEADS}，每头 d_k = {D_MODEL // NUM_HEADS}")

In [ ]:
# 可视化 4 个头的注意力差异
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
words = [id2word[t] for t in token_ids]

for h_idx, ax in enumerate(axes.flat):
    w = head_weights[h_idx]
    im = ax.imshow(w, cmap='Blues', vmin=0, vmax=w.max())
    ax.set_title(f'Head {h_idx+1}（d_k={D_MODEL//NUM_HEADS}）', fontsize=12)
    ax.set_xticks(range(SEQ_LEN))
    ax.set_yticks(range(SEQ_LEN))
    ax.set_xticklabels(words, fontsize=9, rotation=30)
    ax.set_yticklabels(words, fontsize=9)
    ax.set_xlabel('Key（被关注）')
    ax.set_ylabel('Query（主动关注）')
    for i in range(SEQ_LEN):
        for j in range(SEQ_LEN):
            if w[i,j] > 0.005:
                ax.text(j, i, f'{w[i,j]:.2f}', ha='center', va='center', fontsize=8,
                        color='white' if w[i,j] > 0.35 else 'black')
    plt.colorbar(im, ax=ax)

plt.suptitle(f'多头注意力：{NUM_HEADS} 个头对同一序列的不同关注模式\n（随机权重，仅演示形状；训练后每头将捕获不同语言模式）', 
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('multi_head_attention.png', dpi=100, bbox_inches='tight')
plt.show()
print("\n每个头的注意力模式不同——这是多头注意力的核心价值。")
print("真实的预训练模型中，研究者已经发现各头分别捕获了语法、共指、局部等不同模式。")

## 第七章：前馈网络（Feed-Forward Network, FFN）

### 7.1 FFN 的角色

注意力层负责**信息聚合**（把相关位置的信息汇集到当前位置）。
FFN 负责**信息变换**（在每个位置独立地进行特征变换）。

$$\text{FFN}(x) = \text{Activation}(xW_1 + b_1)W_2 + b_2$$

- $W_1 \in \mathbb{R}^{d_{model} \times d_{ff}}$，$d_{ff} = 4 \times d_{model}$（论文原版）
- $W_2 \in \mathbb{R}^{d_{ff} \times d_{model}}$
- **位置独立**：每个 token 的 FFN 输入/输出完全独立，不需要看其他位置

### 7.2 为什么中间维度是 4×？

这是经验性发现。有理论认为：
- **FFN 是「记忆库」**：Geva et al. (2021) 的研究表明，FFN 中的神经元存储了大量「事实知识」（如「巴黎是法国首都」）
- 4× 中间维度提供足够的存储容量
- 现代模型（LLaMA、Mistral）通常用 8/3 × d_model 配合 SwiGLU 激活

### 7.3 激活函数演进

| 激活 | 公式 | 使用模型 |
|------|------|---------|
| ReLU | max(0, x) | 原始 Transformer |
| GeLU | ~x·Φ(x) | GPT-2/3, BERT |
| SwiGLU | Swish(x)·(xW+b) | LLaMA, PaLM |

**GeLU 的优势**：对负数有平滑的非零梯度，训练更稳定；类似 ReLU 但「更柔和」。

**SwiGLU 的优势**：门控机制，当前位置的激活值由另一个线性变换控制，表达能力更强。

### 7.4 FFN 参数占比

- Attention 参数：$4 \times d_{model}^2$（Q, K, V, O）
- FFN 参数：$2 \times d_{model} \times d_{ff} = 2 \times d_{model} \times 4d_{model} = 8 \times d_{model}^2$
- **FFN 参数是 Attention 的 2 倍！** 大部分模型参数在 FFN 中。

In [ ]:
# ---- 激活函数实现 ----
def relu(x):
    return np.maximum(0, x)

def gelu(x):
    """GeLU 近似（论文推荐的 tanh 近似版本）"""
    return 0.5 * x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))

def swish(x):
    return x / (1 + np.exp(-x))

def swiglu(x, W_gate, W_up, W_down):
    """SwiGLU FFN（LLaMA 风格）"""
    gate = swish(x @ W_gate)
    up   = x @ W_up
    return (gate * up) @ W_down

# ---- 标准 FFN（GeLU 版）----
def ffn(x, W1, b1, W2, b2, activation='gelu'):
    act = gelu if activation == 'gelu' else relu
    return act(x @ W1 + b1) @ W2 + b2

# ---- 可视化激活函数 ----
x_range = np.linspace(-3, 3, 300)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：函数值
axes[0].plot(x_range, relu(x_range), label='ReLU', linewidth=2.5, color='#e74c3c')
axes[0].plot(x_range, gelu(x_range), label='GeLU', linewidth=2.5, color='#3498db', linestyle='--')
axes[0].plot(x_range, swish(x_range), label='Swish', linewidth=2.5, color='#2ecc71', linestyle=':')
axes[0].axhline(0, color='k', linewidth=0.5)
axes[0].axvline(0, color='k', linewidth=0.5)
axes[0].set_title('激活函数对比', fontsize=13)
axes[0].legend(fontsize=12)
axes[0].grid(alpha=0.3)
axes[0].set_xlabel('输入 x')
axes[0].set_ylabel('输出 f(x)')

# 右图：梯度（导数）
dx = x_range[1] - x_range[0]
axes[1].plot(x_range[1:], np.diff(relu(x_range))/dx, label='ReLU 梯度', linewidth=2.5, color='#e74c3c')
axes[1].plot(x_range[1:], np.diff(gelu(x_range))/dx, label='GeLU 梯度', linewidth=2.5, color='#3498db', linestyle='--')
axes[1].plot(x_range[1:], np.diff(swish(x_range))/dx, label='Swish 梯度', linewidth=2.5, color='#2ecc71', linestyle=':')
axes[1].axhline(0, color='k', linewidth=0.5)
axes[1].axhline(1, color='k', linewidth=0.5, linestyle='--', alpha=0.3)
axes[1].set_title('梯度对比\n（ReLU 在 x<0 梯度为0，可能导致"神经元死亡"）', fontsize=13)
axes[1].legend(fontsize=12)
axes[1].grid(alpha=0.3)
axes[1].set_ylim(-0.2, 1.3)
axes[1].set_xlabel('输入 x')
axes[1].set_ylabel('梯度 df/dx')

plt.tight_layout()
plt.savefig('activation_functions.png', dpi=120, bbox_inches='tight')
plt.show()

# ---- 参数量对比 ----
d_model = 512
d_ff = 4 * d_model
print(f"d_model={d_model}, d_ff={d_ff}")
print(f"Attention 参数量: {4*d_model*d_model:,}  (Q,K,V,O 各 {d_model}x{d_model})")
print(f"FFN 参数量:       {2*d_model*d_ff:,}  (W1:{d_model}x{d_ff}, W2:{d_ff}x{d_model})")
print(f"FFN 是 Attention 参数的 {2*d_model*d_ff/(4*d_model*d_model):.1f} 倍")

## 第八章：层归一化（Layer Normalization）

### 8.1 为什么需要归一化？

深层网络训练时存在**内部协变量偏移**问题：
- 前一层的参数变化会导致后一层输入分布改变
- 每层都在不断适应变化的输入，训练不稳定

**批归一化（Batch Norm）** 对整个 batch 在特征维度做归一化。
但 NLP 中序列长度不定、batch size 小，Batch Norm 统计量不可靠。

**层归一化（Layer Norm）** 对每个样本的特征维度做归一化，不依赖 batch：

$$\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

- $\mu$：该 token 在 d_model 维度上的均值
- $\sigma^2$：方差
- $\gamma, \beta$：可学习的仿射变换参数（初始化为 1 和 0）
- $\epsilon = 10^{-5}$：防止除以 0

### 8.2 Pre-LN vs Post-LN

**Post-LN（论文原版）：**
$$x = \text{LayerNorm}(x + \text{Sublayer}(x))$$

**Pre-LN（GPT-2 起广泛采用）：**
$$x = x + \text{Sublayer}(\text{LayerNorm}(x))$$

Pre-LN 的优势：梯度直接通过残差连接流向早期层，训练更稳定，不需要 warm-up 学习率策略。

### 8.3 RMSNorm（LLaMA 使用）

$$\text{RMSNorm}(x) = \frac{x}{\text{RMS}(x)} \cdot \gamma, \quad \text{RMS}(x) = \sqrt{\frac{1}{d}\sum x_i^2}$$

省去了均值减法，计算更快，效果相当。

In [ ]:
def layer_norm(x, gamma, beta, eps=1e-5):
    """
    层归一化
    x: (seq_len, d_model)
    gamma, beta: (d_model,)
    """
    mean = x.mean(axis=-1, keepdims=True)    # 沿特征维度求均值
    var  = x.var(axis=-1, keepdims=True)     # 沿特征维度求方差
    x_normalized = (x - mean) / np.sqrt(var + 1e-5)
    return gamma * x_normalized + beta       # 可学习的仿射变换

def rms_norm(x, gamma, eps=1e-5):
    """RMSNorm（LLaMA 使用）——更简洁，省去均值中心化"""
    rms = np.sqrt((x**2).mean(axis=-1, keepdims=True) + eps)
    return gamma * (x / rms)

# ---- 演示：归一化前后的分布变化 ----
np.random.seed(42)
# 故意构造一个分布偏斜的输入（模拟深层网络中可能出现的情况）
x_demo = np.random.randn(SEQ_LEN, D_MODEL) * 5 + 3  

gamma_ln = np.ones(D_MODEL)   # 初始化为1
beta_ln  = np.zeros(D_MODEL)  # 初始化为0

x_ln = layer_norm(x_demo, gamma_ln, beta_ln)
x_rms = rms_norm(x_demo, gamma_ln)

print("输入统计（每个 token）:")
print(f"{'Token':<12} {'均值(原始)':>10} {'标准差(原始)':>12} {'均值(LN后)':>10} {'标准差(LN后)':>12}")
print("-" * 60)
for i, word in enumerate([id2word[t] for t in token_ids]):
    print(f"{word:<12} {x_demo[i].mean():>10.3f} {x_demo[i].std():>12.3f} "
          f"{x_ln[i].mean():>10.5f} {x_ln[i].std():>12.5f}")

print("\n归一化后：每个 token 的均值≈0，标准差≈1")
print("(当 gamma=1, beta=0 时，LN 将分布标准化到 N(0,1))")
print()
print("注：实际训练时，gamma 和 beta 会被学习，使模型可以调整归一化后的分布范围")

## 第九章：残差连接（Residual Connection）

### 9.1 深层网络的梯度消失问题

没有残差连接时，梯度从输出层反向传播到输入层需要经过**所有层**的矩阵乘法：

$$\frac{\partial L}{\partial x_0} = \frac{\partial L}{\partial x_N} \cdot \prod_{l=1}^{N} \frac{\partial x_l}{\partial x_{l-1}}$$

如果每层的雅可比矩阵的奇异值小于 1，梯度会指数衰减（梯度消失）。
GPT-3 有 96 层，不加残差连接根本无法训练。

### 9.2 残差连接的解决方案

$$x_{l+1} = x_l + F_l(x_l)$$

梯度反向传播时：
$$\frac{\partial x_{l+1}}{\partial x_l} = I + \frac{\partial F_l(x_l)}{\partial x_l}$$

**单位矩阵 I 保证梯度至少有「直接通路」**，不会因层数增加而消失。

可以把整个网络理解为：学习**「残差」**而非直接的映射函数。
初始化时 F_l ≈ 0，网络是恒等变换，训练逐步让每层学习残差修正。

### 9.3 Pre-LN 残差结构（现代 GPT 标准）

```python
# 子层 1：注意力
x = x + Dropout(MHA(LayerNorm(x)))

# 子层 2：FFN
x = x + Dropout(FFN(LayerNorm(x)))
```

**Dropout** 在训练时随机将部分激活归零（概率 p，通常 0.1），
防止过拟合，推理时不使用。

In [ ]:
# ---- 演示残差连接的梯度流动 ----
# 用一个简单的深层网络对比：有/无残差

def layer_no_residual(x, W):
    """无残差：x_out = tanh(x @ W)"""
    return np.tanh(x @ W)

def layer_with_residual(x, W):
    """有残差：x_out = x + tanh(x @ W)"""
    return x + np.tanh(x @ W)

np.random.seed(0)
dim = 8
num_layers = 20

# 用小权重（接近恒等变换的初始化）
weights = [np.random.randn(dim, dim) * 0.1 for _ in range(num_layers)]

# 前向传播，记录每层输出的范数
x_init = np.random.randn(dim)

x_no_res = x_init.copy()
x_with_res = x_init.copy()
norms_no_res = [np.linalg.norm(x_no_res)]
norms_with_res = [np.linalg.norm(x_with_res)]

for W in weights:
    x_no_res   = layer_no_residual(x_no_res, W)
    x_with_res = layer_with_residual(x_with_res, W)
    norms_no_res.append(np.linalg.norm(x_no_res))
    norms_with_res.append(np.linalg.norm(x_with_res))

fig, ax = plt.subplots(figsize=(10, 5))
layers = range(len(norms_no_res))
ax.plot(layers, norms_no_res,   'o-', color='#e74c3c', label='无残差连接', linewidth=2, markersize=4)
ax.plot(layers, norms_with_res, 's-', color='#3498db', label='有残差连接', linewidth=2, markersize=4)
ax.set_xlabel('网络层数', fontsize=12)
ax.set_ylabel('激活值范数', fontsize=12)
ax.set_title(f'残差连接对 {num_layers} 层网络激活值的影响\n（有残差的信号更稳定，梯度不会消失或爆炸）', fontsize=12)
ax.legend(fontsize=12)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('residual_connection.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"20 层后激活范数:")
print(f"  无残差: {norms_no_res[-1]:.4f}（趋向0，梯度消失）")
print(f"  有残差: {norms_with_res[-1]:.4f}（保持合理范围）")

## 第十章：完整 Decoder Block

现在把所有组件组装成一个完整的 GPT 风格 Decoder Block：

```
输入 x (seq_len, d_model)
│
├── LayerNorm(x)
│       │
│       └── MultiHeadAttention（因果掩码）
│                   │
└──────────────── + ──→ x1 = x + MHA(LN(x))
                              │
                    ├── LayerNorm(x1)
                    │       │
                    │       └── FFN
                    │               │
                    └────────── + ──→ x2 = x1 + FFN(LN(x1))
                                          │
                                        输出
```

In [ ]:
class DecoderBlock:
    """
    完整的 GPT Decoder Block（NumPy 实现，Pre-LN 风格）
    
    包含：Masked Self-Attention + FFN + 两个 LayerNorm + 残差连接
    """
    
    def __init__(self, d_model, num_heads, d_ff, seed=0):
        np.random.seed(seed)
        s = 0.02  # 小权重初始化（类似 GPT-2）
        
        # MHA 权重
        self.W_q = np.random.randn(d_model, d_model) * s
        self.W_k = np.random.randn(d_model, d_model) * s
        self.W_v = np.random.randn(d_model, d_model) * s
        self.W_o = np.random.randn(d_model, d_model) * s
        
        # FFN 权重（GeLU 激活）
        self.W1 = np.random.randn(d_model, d_ff) * s
        self.b1 = np.zeros(d_ff)
        self.W2 = np.random.randn(d_ff, d_model) * s
        self.b2 = np.zeros(d_model)
        
        # LayerNorm 参数（γ=1, β=0 初始化）
        self.gamma1 = np.ones(d_model);  self.beta1 = np.zeros(d_model)
        self.gamma2 = np.ones(d_model);  self.beta2 = np.zeros(d_model)
        
        self.num_heads = num_heads
        self.d_model = d_model
    
    def forward(self, x):
        """
        x: (seq_len, d_model)
        返回: (seq_len, d_model) -- 形状不变
        """
        seq_len = x.shape[0]
        causal_mask = make_causal_mask(seq_len)
        
        # ===== 子层1：Masked Self-Attention + 残差 =====
        x_norm = layer_norm(x, self.gamma1, self.beta1)                # Pre-LN
        attn_out, attn_weights = multi_head_attention(
            x_norm, self.W_q, self.W_k, self.W_v, self.W_o,
            self.num_heads, mask=causal_mask
        )
        x = x + attn_out                                               # 残差
        
        # ===== 子层2：FFN + 残差 =====
        x_norm = layer_norm(x, self.gamma2, self.beta2)                # Pre-LN
        ffn_out = ffn(x_norm, self.W1, self.b1, self.W2, self.b2, activation='gelu')
        x = x + ffn_out                                                # 残差
        
        return x, attn_weights

# ---- 测试单个 Block ----
d_model_big = 32
num_heads    = 4
d_ff_big     = 4 * d_model_big
seq_len_test = 6

# 构建输入
x_test_input = np.random.randn(seq_len_test, d_model_big)
block = DecoderBlock(d_model_big, num_heads, d_ff_big, seed=42)
x_out, _ = block.forward(x_test_input)

print(f"Decoder Block 测试:")
print(f"  输入 shape: {x_test_input.shape}")
print(f"  输出 shape: {x_out.shape}  （shape 完全不变）")
print(f"  输入 L2 范数: {np.linalg.norm(x_test_input):.4f}")
print(f"  输出 L2 范数: {np.linalg.norm(x_out):.4f}")

## 第十一章：堆叠多个 Block + 输出头 = 完整 Mini-GPT

现在构建一个完整的（小型）GPT：
- 堆叠 N 个 Decoder Block
- 最后加一个「输出头」：Linear + Softmax → 词表上的概率分布
- 实现自回归生成（贪心、温度采样、Top-k、Top-p）

In [ ]:
class MiniGPT:
    """
    完整的 GPT 模型（NumPy 实现）
    
    架构：
      Token Embedding + Positional Encoding
      → N × Decoder Block
      → Final LayerNorm
      → Linear（d_model → vocab_size）
      → Softmax → 概率分布
    """
    
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_seq_len):
        np.random.seed(99)
        s = 0.02
        
        self.vocab_size  = vocab_size
        self.d_model     = d_model
        self.max_seq_len = max_seq_len
        
        # Token Embedding 表
        self.token_emb = np.random.randn(vocab_size, d_model) * s
        
        # Decoder Blocks
        self.blocks = [DecoderBlock(d_model, num_heads, d_ff, seed=i)
                       for i in range(num_layers)]
        
        # 最终 LayerNorm
        self.gamma_f = np.ones(d_model)
        self.beta_f  = np.zeros(d_model)
        
        # 输出头（Language Model Head）：映射到词表大小
        self.lm_head = np.random.randn(d_model, vocab_size) * s
    
    def forward(self, token_ids):
        """
        token_ids: list of int，长度为 seq_len
        返回: logits (seq_len, vocab_size)
        """
        seq_len = len(token_ids)
        assert seq_len <= self.max_seq_len
        
        # Step 1: Token Embedding 查表
        x = self.token_emb[token_ids]   # (seq_len, d_model)
        
        # Step 2: 叠加位置编码
        x = x + positional_encoding(seq_len, self.d_model)
        
        # Step 3: 逐层 Decoder Block
        for block in self.blocks:
            x, _ = block.forward(x)
        
        # Step 4: 最终 LayerNorm
        x = layer_norm(x, self.gamma_f, self.beta_f)
        
        # Step 5: 输出头 → logits
        logits = x @ self.lm_head   # (seq_len, vocab_size)
        
        return logits
    
    def get_next_token_probs(self, token_ids, temperature=1.0):
        """获取下一个 token 的概率分布"""
        logits = self.forward(token_ids)
        last_logits = logits[-1]   # 取最后一个位置的 logits
        
        if temperature != 1.0:
            last_logits = last_logits / temperature   # 温度缩放
        
        probs = softmax(last_logits)
        return probs

# 构建一个小 GPT
model = MiniGPT(
    vocab_size  = VOCAB_SIZE,
    d_model     = 32,
    num_heads   = 4,
    d_ff        = 128,
    num_layers  = 3,
    max_seq_len = 20
)

# 前向传播
input_ids = [vocab['<BOS>'], vocab['the'], vocab['cat']]
logits = model.forward(input_ids)
print(f"输入 token IDs: {input_ids} → {[id2word[t] for t in input_ids]}")
print(f"Logits shape: {logits.shape}  (每个位置对词表中每个词的分数)")
print(f"\n最后位置（'cat'）预测下一词的概率分布 Top-5:")
probs = softmax(logits[-1])
top5 = np.argsort(probs)[::-1][:5]
for rank, idx in enumerate(top5):
    print(f"  {rank+1}. '{id2word[idx]}': {probs[idx]:.4f}")
print("\n注：当前是随机初始化权重，所以概率分布接近均匀；训练后概率会集中在合理的下一词")

## 第十二章：自回归文本生成

语言模型生成文本的方式：**每次预测一个 token，将其追加到输入，再预测下一个**。

### 12.1 贪心搜索（Greedy Search）
每次选概率最高的 token。
- 优点：确定性，快
- 缺点：容易陷入重复循环，缺乏多样性

### 12.2 温度采样（Temperature Sampling）
$$p_i \leftarrow \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$
- T=1：正常概率
- T<1（如 0.5）：分布「更尖」，倾向高概率词，更保守
- T>1（如 2.0）：分布「更平」，随机性更大，更有创意
- T→0：退化为贪心搜索

### 12.3 Top-k 采样
每次只从概率最高的 k 个 token 中采样（丢弃其余词）。
防止低概率词意外被选到（如 k=50）。

### 12.4 Top-p 采样（Nucleus Sampling）
将 token 按概率从高到低排列，找到使累积概率 ≥ p 的最小集合，从该集合中采样。
- 动态调整候选词数量（高确定性时候选词少，低确定性时候选词多）
- 通常 p=0.9 或 0.95

**实际应用（Claude/GPT API）：** 一般同时使用 temperature + top-p：
```python
temperature=0.7, top_p=0.9  # 常见设置
```

In [ ]:
def greedy_sample(probs):
    """贪心：选概率最高的 token"""
    return int(np.argmax(probs))

def temperature_sample(probs, temperature=1.0):
    """温度采样"""
    if temperature == 0:
        return greedy_sample(probs)
    # 注意：probs 已经是 softmax 后的，重新从 logits 角度缩放
    logits = np.log(probs + 1e-10)
    logits = logits / temperature
    scaled_probs = softmax(logits)
    return int(np.random.choice(len(scaled_probs), p=scaled_probs))

def top_k_sample(probs, k=5):
    """Top-k 采样：只在 k 个最高概率词中采样"""
    top_k_idx = np.argsort(probs)[-k:]          # 找 top-k 索引
    top_k_probs = probs[top_k_idx]
    top_k_probs = top_k_probs / top_k_probs.sum() # 重归一化
    chosen = np.random.choice(len(top_k_probs), p=top_k_probs)
    return int(top_k_idx[chosen])

def top_p_sample(probs, p=0.9):
    """Top-p（Nucleus）采样"""
    sorted_idx   = np.argsort(probs)[::-1]       # 从高到低排序
    sorted_probs = probs[sorted_idx]
    cumulative   = np.cumsum(sorted_probs)
    # 找到累积概率首次超过 p 的位置
    cutoff = np.searchsorted(cumulative, p) + 1
    nucleus_idx   = sorted_idx[:cutoff]
    nucleus_probs = probs[nucleus_idx]
    nucleus_probs = nucleus_probs / nucleus_probs.sum()
    chosen = np.random.choice(len(nucleus_idx), p=nucleus_probs)
    return int(nucleus_idx[chosen])

def generate(model, prompt_ids, max_new_tokens=10, strategy='greedy', 
             temperature=1.0, top_k=5, top_p=0.9, eos_id=2):
    """自回归生成"""
    generated = list(prompt_ids)
    for _ in range(max_new_tokens):
        probs = model.get_next_token_probs(generated, temperature=temperature)
        if strategy == 'greedy':
            next_token = greedy_sample(probs)
        elif strategy == 'temperature':
            next_token = temperature_sample(probs, temperature)
        elif strategy == 'top_k':
            next_token = top_k_sample(probs, top_k)
        elif strategy == 'top_p':
            next_token = top_p_sample(probs, top_p)
        generated.append(next_token)
        if next_token == eos_id:
            break
    return generated

# ---- 对比不同采样策略 ----
prompt = [vocab['<BOS>'], vocab['the']]
prompt_text = ' '.join(id2word[t] for t in prompt)

print(f"提示词: {prompt_text}\n")
np.random.seed(42)

strategies = [
    ('greedy',      dict()),
    ('temperature', dict(temperature=0.5)),
    ('temperature', dict(temperature=2.0)),
    ('top_k',       dict(temperature=1.0, top_k=4)),
    ('top_p',       dict(temperature=1.0, top_p=0.8)),
]

for strat, kwargs in strategies:
    result = generate(model, prompt, max_new_tokens=5, strategy=strat, **kwargs)
    text = ' '.join(id2word[t] for t in result)
    params = ', '.join(f'{k}={v}' for k,v in kwargs.items()) if kwargs else ''
    print(f"  [{strat:12s}] {params:22s} → {text}")

print("\n注：当前是随机权重，生成结果无意义。训练后的模型会生成有语义的句子。")

## 第十三章：训练目标——Next Token Prediction

### 13.1 自监督学习

GPT 的训练不需要人工标注数据！
只需要大量文本，每个样本自动生成：

```
文本: "the cat sat on the mat"
输入:     ["the", "cat", "sat", "on",  "the"]
标签:     ["cat", "sat", "on",  "the", "mat"]
```

对每个位置，用**交叉熵损失**衡量预测分布和真实 token 之间的差距：

$$L = -\frac{1}{N}\sum_{t=1}^{N} \log p(x_t | x_1, ..., x_{t-1})$$

### 13.2 交叉熵损失的直觉

如果模型对真实 token 的预测概率很高（如 0.9），损失 = -log(0.9) ≈ 0.1（小）
如果预测概率很低（如 0.01），损失 = -log(0.01) ≈ 4.6（大）

完美预测时损失 = 0；随机猜测时损失 ≈ log(vocab_size)

### 13.3 困惑度（Perplexity）

语言模型常用**困惑度**（Perplexity, PPL）衡量：

$$\text{PPL} = e^L = e^{-\frac{1}{N}\sum \log p(x_t|x_{<t})}$$

直觉：PPL 是模型「有效词表大小」的估计。
- PPL=10：每次选择等价于从 10 个词中猜
- PPL=1000：等价于从 1000 个词中随机猜（很差）
- GPT-4 在某些 benchmark 上 PPL < 5

In [ ]:
def cross_entropy_loss(logits, target_ids):
    """
    计算序列的交叉熵损失
    logits: (seq_len, vocab_size)
    target_ids: (seq_len,) 每个位置的真实 token id
    """
    seq_len = len(target_ids)
    probs = softmax(logits, axis=-1)   # (seq_len, vocab_size)
    
    # 取每个位置真实 token 的预测概率
    target_probs = probs[np.arange(seq_len), target_ids]  # (seq_len,)
    
    # 交叉熵 = -log(p)
    loss_per_token = -np.log(target_probs + 1e-10)
    
    return loss_per_token.mean(), loss_per_token

def perplexity(loss):
    return np.exp(loss)

# ---- 演示训练目标 ----
# 句子: <BOS> the cat sat on mat
# 输入: [<BOS>, the, cat, sat, on]   → 预测 → [the, cat, sat, on, mat]
input_ids_train  = token_ids[:-1]   # 去掉最后一个
target_ids_train = token_ids[1:]    # 去掉第一个

print("训练样本构造:")
print(f"  输入:  {[id2word[t] for t in input_ids_train]}")
print(f"  标签:  {[id2word[t] for t in target_ids_train]}")
print(f"  (每个输入位置预测对应的下一个词)\n")

logits_train = model.forward(input_ids_train)
loss, loss_per_tok = cross_entropy_loss(logits_train, target_ids_train)
ppl = perplexity(loss)

print(f"初始（随机权重）损失: {loss:.4f}")
print(f"初始困惑度 (PPL):     {ppl:.1f}")
print(f"理论随机猜测损失:     {np.log(VOCAB_SIZE):.4f}  (log({VOCAB_SIZE}))")
print(f"理论随机猜测 PPL:     {VOCAB_SIZE}")
print()
print("逐 token 损失:")
for i, (t_in, t_out, l) in enumerate(zip(
        [id2word[t] for t in input_ids_train],
        [id2word[t] for t in target_ids_train],
        loss_per_tok)):
    print(f"  位置{i}: '{t_in}' → 预测 '{t_out}'，损失={l:.3f}")

# 可视化损失
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(range(len(loss_per_tok)), loss_per_tok, 
            color=['#e74c3c' if l > loss else '#3498db' for l in loss_per_tok])
axes[0].axhline(loss, color='black', linestyle='--', label=f'平均损失={loss:.3f}')
axes[0].axhline(np.log(VOCAB_SIZE), color='gray', linestyle=':', label=f'随机基线={np.log(VOCAB_SIZE):.3f}')
axes[0].set_xticks(range(len(loss_per_tok)))
axes[0].set_xticklabels([f"'{id2word[input_ids_train[i]]}'→'{id2word[target_ids_train[i]]}'" 
                          for i in range(len(loss_per_tok))], rotation=30, ha='right')
axes[0].set_ylabel('交叉熵损失')
axes[0].set_title('每个 token 位置的预测损失\n（红色=高于平均，蓝色=低于平均）')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# 温度对概率分布的影响
temps = [0.3, 0.7, 1.0, 1.5, 2.0]
last_probs = [softmax(logits_train[-1] / T) for T in temps]
for temp, p in zip(temps, last_probs):
    sorted_p = np.sort(p)[::-1][:8]
    axes[1].plot(range(len(sorted_p)), sorted_p, 'o-', label=f'T={temp}', linewidth=1.5, markersize=5)
axes[1].set_xlabel('Token 排名（按概率从高到低）')
axes[1].set_ylabel('概率')
axes[1].set_title('不同温度对输出概率分布的影响\n（T小→尖锐集中，T大→平坦分散）')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_objective.png', dpi=120, bbox_inches='tight')
plt.show()

## 第十四章：用 PyTorch 验证

上面用 NumPy 手动实现了所有组件。PyTorch 有官方实现，用于实际训练。
下面验证我们对架构的理解，并展示 GPT-2 的实际参数结构。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---- 构建一个标准的 GPT-2 Small 配置（但规模缩小用于演示）----
class GPTConfig:
    vocab_size  = 50257  # GPT-2 真实词表大小
    d_model     = 128    # 演示用（GPT-2 Small 真实是 768）
    num_heads   = 4      # 演示用（GPT-2 Small 真实是 12）
    num_layers  = 2      # 演示用（GPT-2 Small 真实是 12）
    d_ff        = 512    # 演示用（GPT-2 Small 真实是 3072）
    max_seq_len = 64     # 演示用（GPT-2 真实是 1024）
    dropout     = 0.0    # 演示关闭 dropout

cfg = GPTConfig()

class GPTBlock(nn.Module):
    """标准 GPT Decoder Block（PyTorch）"""
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = nn.MultiheadAttention(cfg.d_model, cfg.num_heads, 
                                           batch_first=True, dropout=cfg.dropout)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn  = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.d_ff),
            nn.GELU(),
            nn.Linear(cfg.d_ff, cfg.d_model),
        )
    
    def forward(self, x):
        seq_len = x.shape[1]
        # 因果掩码
        causal_mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)
        
        # Self-Attention + 残差（Pre-LN）
        x_norm = self.ln1(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm, attn_mask=causal_mask)
        x = x + attn_out
        
        # FFN + 残差（Pre-LN）
        x = x + self.ffn(self.ln2(x))
        return x

class ToyGPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.token_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb   = nn.Embedding(cfg.max_seq_len, cfg.d_model)  # 可学习位置编码
        self.blocks    = nn.ModuleList([GPTBlock(cfg) for _ in range(cfg.num_layers)])
        self.ln_f      = nn.LayerNorm(cfg.d_model)
        self.lm_head   = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
    
    def forward(self, token_ids):
        B, T = token_ids.shape
        pos   = torch.arange(T).unsqueeze(0)
        x     = self.token_emb(token_ids) + self.pos_emb(pos)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return self.lm_head(x)   # (B, T, vocab_size)

model_torch = ToyGPT(cfg)

# ---- 参数统计 ----
total_params = sum(p.numel() for p in model_torch.parameters())
trainable    = sum(p.numel() for p in model_torch.parameters() if p.requires_grad)
print(f"模型配置: d_model={cfg.d_model}, heads={cfg.num_heads}, layers={cfg.num_layers}")
print(f"总参数量: {total_params:,}")
print(f"可训练参数: {trainable:,}")
print()

# 参数分解
for name, param in model_torch.named_parameters():
    print(f"  {name:<40} {str(param.shape):<25} {param.numel():>10,}")

# 前向传播测试
dummy_input = torch.randint(0, cfg.vocab_size, (1, 10))  # batch=1, seq=10
with torch.no_grad():
    output = model_torch(dummy_input)
print(f"\n前向传播: 输入 {dummy_input.shape} → 输出 {output.shape}")
print(f"  输出的最后维度={cfg.vocab_size}（词表大小，每个位置预测下一词的 logits）")

In [ ]:
# ---- GPT-2 真实规模对比 ----
print("=" * 55)
print("GPT 系列模型规模对比")
print("=" * 55)

gpt_configs = [
    ("GPT-2 Small",  768,  12, 12, 3072,  117e6),
    ("GPT-2 Medium", 1024, 16, 24, 4096,  345e6),
    ("GPT-2 Large",  1280, 20, 36, 5120,  774e6),
    ("GPT-2 XL",     1600, 25, 48, 6400,  1542e6),
    ("GPT-3",        12288,96, 96, 49152, 175e9),
]

print(f"{'模型':<15} {'d_model':>8} {'heads':>6} {'layers':>7} {'d_ff':>7} {'参数量':>12}")
print("-" * 60)
for name, dm, nh, nl, dff, params in gpt_configs:
    p_str = f"{params/1e6:.0f}M" if params < 1e9 else f"{params/1e9:.0f}B"
    print(f"{name:<15} {dm:>8} {nh:>6} {nl:>7} {dff:>7} {p_str:>12}")

print()
print("每个 Decoder Layer 的参数量（以 GPT-2 Small 为例）：")
dm = 768
nh = 12
dff = 4 * dm
# MHA：Q,K,V,O 各 dm×dm，加上 bias
mha = 4 * dm * dm + 4 * dm
# FFN：两个线性层
ffn_p = dm * dff + dff + dff * dm + dm
# LN：两个，各 2×dm
ln_p = 2 * 2 * dm
total_per_layer = mha + ffn_p + ln_p
print(f"  MHA:         {mha:>10,}  ({mha/total_per_layer*100:.1f}%)")
print(f"  FFN:         {ffn_p:>10,}  ({ffn_p/total_per_layer*100:.1f}%)")
print(f"  LayerNorm:   {ln_p:>10,}  ({ln_p/total_per_layer*100:.1f}%)")
print(f"  每层合计:    {total_per_layer:>10,}")
print(f"  12 层合计:   {total_per_layer*12:>10,}")
print(f"  含 Emb+Head: ≈ 117,000,000  (GPT-2 Small 官方)")

## 第十五章：扩展阅读与下一步

### 重要论文（按难度排序）

1. **[Attention Is All You Need](https://arxiv.org/abs/1706.03762)** (2017)
   - Transformer 原始论文，必读
   
2. **[Language Models are Unsupervised Multitask Learners](https://d4mucfpksywv.cloudfront.net/better-language-models/language-models.pdf)** (GPT-2, 2019)
   - 展示 scale 带来的 emergent ability
   
3. **[An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale](https://arxiv.org/abs/2010.11929)** (ViT, 2020)
   - Transformer 在 CV 的成功应用

4. **[Scaling Laws for Neural Language Models](https://arxiv.org/abs/2001.08361)** (2020)
   - 规模定律，指导如何分配计算资源

5. **[RoFormer: Enhanced Transformer with Rotary Position Embedding](https://arxiv.org/abs/2104.09864)** (RoPE, 2021)
   - LLaMA 使用的旋转位置编码

### 优秀实现资源

- **nanoGPT**（Andrej Karpathy）：300 行代码实现 GPT-2，最佳学习资源
- **The Annotated Transformer**（哈佛 NLP）：带注释的完整 Transformer 实现
- **minGPT**：教学版 GPT，带详细注释

### 下一步学习路径

```
本节（Transformer 架构）
    ↓
02_tokenization.ipynb（分词与 BPE 算法）
    ↓
03_prompting.ipynb（如何有效使用 LLM）
    ↓
04_rag_intro.ipynb（检索增强生成）
    ↓
02_agents_frameworks/（LangChain、AutoGen、Claude SDK）
```

## 本节总结

| 组件 | 解决的问题 | 关键公式/参数 |
|------|-----------|-------------|
| **位置编码** | Attention 置换不变 | sin/cos，10000^(2i/d) |
| **Self-Attention** | 全局依赖，O(1) 路径 | softmax(QK^T/√d_k)V |
| **因果掩码** | 防止信息泄露 | 上三角 → -∞ |
| **Multi-Head** | 多种依赖模式 | h 个子空间并行 |
| **FFN** | 位置独立变换 | 4×d_model 中间层，GeLU |
| **LayerNorm** | 训练稳定性 | Pre-LN 更优 |
| **残差连接** | 深层梯度流动 | x = x + F(x) |
| **温度采样** | 生成多样性控制 | logits / T |
| **Top-p 采样** | 动态候选词集合 | Nucleus Sampling |
| **交叉熵损失** | 训练目标 | -log P(x_t \| x_{<t}) |